# Stim `SELECT` blocks

In [ ]:
from qdk import stim
from qdk.widgets import Histogram

## Selecting a measurement outcome

In [ ]:
select_zero = """
SELECT {
    H 0
    M 0
    REQUIRE rec[-1]
}
"""

# REQUIRE rec[-1] restarts while M 0 == 1, so every shot reports 0.
Histogram(stim.run(select_zero, shots=2000, type="clifford"), labels="kets")


## Negating a requirement with `!`


In [ ]:
select_one = """
SELECT {
    H 0
    M 0
    REQUIRE !rec[-1]
}
"""

# Negating with `!` flips the condition: restart while M 0 == 0, so every shot reports 1.
Histogram(stim.run(select_one, shots=2000, type="clifford"), labels="kets")


## Multiple `REQUIRE` statements

In [ ]:
multi_require = """
SELECT {
    H 0
    M 0
    REQUIRE rec[-1]
    H 1
    M 1
    REQUIRE rec[-1]
}
"""

# Both qubits preselected to 0 → only 00 appears.
Histogram(stim.run(multi_require, shots=2000, type="clifford"), labels="kets")

## Parity over several records

In [ ]:
parity_check = """
SELECT {
    R 0
    R 1
    H 0
    H 1
    M 0
    M 1
    REQUIRE rec[-1] rec[-2]
}
"""

# Even parity enforced → only 00 and 11 survive.
Histogram(stim.run(parity_check, shots=2000, type="clifford"), labels="kets")

## Rejecting lost qubits with `NOTLEAKED`

`NOTLEAKED` is the loss counterpart of `REQUIRE`: it restarts the enclosing `SELECT` block whenever a referenced measurement's qubit was *lost* instead of measured. Without it, `LOSS_ERROR` would make some shots report `L`; `NOTLEAKED` discards those so every reported shot has a genuine `0`/`1` outcome.


In [ ]:
survived = """
SELECT {
    H 0
    LOSS_ERROR(0.3) 0
    MR 0
    NOTLEAKED rec[-1]
}
"""

# NOTLEAKED restarts the block whenever qubit 0 is lost, so no shot reports L.
Histogram(stim.run(survived, shots=2000, type="clifford"), labels="kets")


## Nested `SELECT` blocks

In [ ]:
nested = """
SELECT {
    R 1
    SELECT {
        R 0
        H 0
        M 0
        REQUIRE rec[-1]
    }
    H 1
    M 1
    REQUIRE rec[-1]
}
"""

# Inner block selects qubit 0; outer block selects qubit 1 → only 00.
Histogram(stim.run(nested, shots=2000, type="clifford"), labels="kets")

## Measurement record scoping

When a `SELECT` block restarts, it re-runs **only its own body** — measurements made in an enclosing scope are not repeated. A record is *in scope* if it was produced inside the current block (or an inner one); records from an outer block are *out of scope*.

Since a restart can only change in-scope measurements, every `REQUIRE` / `NOTLEAKED` must reference **at least one** in-scope record. Referencing only out-of-scope records could never change the outcome, so it would loop forever and is rejected at compile time.

You *can*, however, combine an out-of-scope record with an in-scope one: the out-of-scope record acts as a fixed condition that the in-scope measurement is selected against.


In [ ]:
scoping = """
H 0
M 0
SELECT {
    H 1
    M 1
    REQUIRE rec[-1] rec[-2]
}
"""

# rec[-1] (M 1) is in scope; rec[-2] (M 0) is fixed from outside the block.
# Only M 1 is re-rolled on restart, until it matches M 0 → just 00 and 11 survive.
Histogram(stim.run(scoping, shots=2000, type="clifford"), labels="kets")
